# 📌 Traccia: Classificazione - Logistic Regression vs Random Forest con Manifold Learning
- Tipo di problema: Classificazione binaria
- Dataset: Dataset sintetico generato con make_classification: n_samples=1200, n_features=100, n_informative=20, n_redundant=10, Classi bilanciate, random_state=42

1. Pipeline 1:
    - Preprocessing: StandardScaler
    - Riduzione dimensionalità: LocallyLinearEmbedding (LLE, n_components=10)
    - Modello: LogisticRegression (ottimizzazione di C)

2. Pipeline 2:
    - Preprocessing: MinMaxScaler
    - Riduzione dimensionalità: Isomap (n_components=10)
    - Modello: RandomForestClassifier (ottimizzazione di n_estimators e max_depth)

- Metrica di valutazione: AUC-ROC (per valutare la separabilità anche nei casi borderline)

- Valutazione tramite Nested Cross-Validation:
    - Outer CV: 5-fold
    - Inner CV: 3-fold per selezione degli iperparametri

## Generazione del dataset
Genero un dataset sintetico adatto a task di classificazione con 1200 osservazioni, 100 features (di cui 20 informative e 10 ridondanti).

In [41]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples = 1200,
                           n_features = 100,
                           n_informative = 20,
                           n_redundant = 10,
                           random_state = 42)

print('X shape', X.shape)
print('y shape', y.shape)

list(y[:5])

X shape (1200, 100)
y shape (1200,)


[0, 1, 1, 1, 0]

## Dataset splitting

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

## Funzione per cercare i migliori iperparametri per il manifold learning

In [43]:
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
from sklearn.model_selection import ParameterSampler
import numpy as np

def best_manifold(X, y, model, param_grid, metric="ami", n_iter=3,
                                          random_state=42):
    best_score = -np.inf
    best_params = None

    # Crea un campionamento casuale di combinazioni di parametri dalla griglia param_grid. Questo è più efficiente rispetto a esplorare tutte le combinazioni.
    sampler = ParameterSampler(param_grid, n_iter=n_iter, random_state=random_state)

    # per ogni set di parametri generato dal sampler
    for params in sampler:
        try:
            # creo il modello usando i parametri specificati
            embedding = model(**params)
            # eseguo l'algoritmo di embedding per ottenere una nuova rappresentazione dei dati nel nuovo spazio (ad esempio, 2d o 3d), chiamata X_embedded
            X_embedded = embedding.fit_transform(X)

            # silhouette calcolato solo se specificato nei parametri del metodo e se l'output dell'embedding è di due dimensioni
            if metric == "silhouette" and X_embedded.shape[1] == 2:
                score = silhouette_score(X_embedded, y)

            elif metric == "ami":
                score = adjusted_mutual_info_score(y, np.argmax(X_embedded, axis=1))
            else:
                continue

            print(f"Executing function--Params: {params} => {metric.upper()} Score: {score:.4f}")

            if score > best_score:
                best_score = score
                best_params = params

        except Exception as e:
            print(f"Error with params {params}: {e}")

    return {"best_params": best_params, "best_score": best_score}

## Pipeline 1:
   - Preprocessing: StandardScaler
   - Riduzione dimensionalità: LocallyLinearEmbedding (LLE, n_components=10)
   - Modello: LogisticRegression (ottimizzazione di C)


### Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()

# X standardizzato
X_std = standard_scaler.fit_transform(X_train)

### Ricerca dei migliori iperparametri per l'embedding

In [5]:
lle_params_grid = {
    'n_neighbors': [3, 5, 7],
    'n_components': [5, 10, 20]
}

In [27]:
from sklearn.manifold import LocallyLinearEmbedding

lle_result_best_manifold = best_manifold(
    X = X_std,
    y = y_train,
    model = LocallyLinearEmbedding,
    param_grid = lle_params_grid
)

Executing function--Params: {'n_neighbors': 5, 'n_components': 20} => AMI Score: 0.0324
Executing function--Params: {'n_neighbors': 5, 'n_components': 5} => AMI Score: 0.0724
Executing function--Params: {'n_neighbors': 7, 'n_components': 10} => AMI Score: 0.0277


In [28]:
best_score_lle = lle_result_best_manifold['best_score']
best_params_lle = lle_result_best_manifold['best_params']

print("Best score: ", best_score_lle)
print("Best parameters: ", best_params_lle)

Best score:  0.07236024186459884
Best parameters:  {'n_neighbors': 5, 'n_components': 5}


### Creazione dell'embedding con i migliori iperparametri trovati

In [46]:
X_train_embedded_lle = LocallyLinearEmbedding(
    n_neighbors =  best_params_lle['n_neighbors'],
    n_components = best_params_lle['n_components']
).fit_transform(X_std)

print(f"Shape of X_train after transformation: {X_train_embedded_lle.shape}")

X_test_embedded_lle = LocallyLinearEmbedding(
    n_neighbors =  best_params_lle['n_neighbors'],
    n_components = best_params_lle['n_components']
).fit_transform(X_test)

print(f"Shape of X_test after transformation: {X_test_embedded_lle.shape}")

Shape of X_train after transformation: (960, 5)
Shape of X_test after transformation: (240, 5)


## Nested Cross Validation

In [33]:
import pandas as pd
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
)

def nested_cv(model, param_grid, X_train, y_train,
              outer_splits=5, inner_splits=5,
              scoring: list[str] = None,
              random_state=42, verbose=True):

    # Assicurati che `y` sia un array 1D
    if isinstance(y_train, pd.DataFrame):  # Se è un DataFrame Pandas
        y_train = y_train.values.ravel()
    elif isinstance(y_train, pd.Series):  # Se è una Serie Pandas
        y_train = y_train.values
    else:  # Se è un array Numpy
        y_train = np.ravel(y_train)

    # Determina il tipo di task di apprendimento automatico
    est_type = getattr(model, "_estimator_type", None)
    is_clf = (est_type == "classifier")

    if scoring is None:
        if is_clf:
            scoring = ['accuracy', 'roc_auc']
        else:
            scoring = ['r2', 'mae', 'rmse']

    # CROSS-VALIDATION ESTERNA
    outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=random_state)

    # Dizionari per salvare i risultati
    score_results = {metric: [] for metric in scoring}
    all_fold_best_params = []

    for outer_fold, (train_fold_idx, val_idx) in enumerate(outer_cv.split(X_train), 1):
        if verbose:
            print(f"\nPerforming Outer Fold {outer_fold}/{outer_splits}")

        # Usare il metodo .iloc per X, se è un DataFrame
        if isinstance(X_train, pd.DataFrame):
            X_train_fold, X_val = X_train.iloc[train_fold_idx], X_train.iloc[val_idx]
        else:  # Altrimenti usa indicizzazione standard
            X_train_fold, X_val = X_train[train_fold_idx], X_train[val_idx]

        y_train_fold, y_val = y_train[train_fold_idx], y_train[val_idx]

        # --- 3. CICLO DI CROSS-VALIDATION INTERNA (TUNING) ---
        inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
        primary_metric = scoring[0]  # GridSearchCV ottimizza per la prima metrica della lista

        if verbose:
            print(f"Performing GridSearchCV (optimizing for '{primary_metric}')...")

        grid_search = GridSearchCV(model, param_grid, cv=inner_cv, n_jobs=-1, scoring=primary_metric)
        grid_search.fit(X_train_fold, y_train_fold)

        # Salva i migliori parametri per questo fold
        all_fold_best_params.append(grid_search.best_params_)
        if verbose:
            print(f"  Best Params for this fold: {grid_search.best_params_}")

        # --- 4. VALUTAZIONE SUL TEST SET ESTERNO ---
        best_model_for_fold = grid_search.best_estimator_
        y_pred = best_model_for_fold.predict(X_val)

        if verbose:
            print("  Calculating metrics on the outer test set...")

        # Calcola e salva tutte le metriche richieste
        for metric in scoring:
            if metric == 'accuracy':
                score = accuracy_score(y_val, y_pred)
            elif metric == 'roc_auc':
                try:
                    y_score = best_model_for_fold.predict_proba(X_val)[:, 1]
                    score = roc_auc_score(y_val, y_score)
                except (AttributeError, IndexError):
                    score = np.nan # Modello non ha predict_proba o è binario/monoclasse
            elif metric == 'r2':
                score = r2_score(y_val, y_pred)
            elif metric == 'mae':
                score = mean_absolute_error(y_val, y_pred)
            elif metric == 'mse':
                score = mean_squared_error(y_val, y_pred)
            elif metric == 'rmse':
                score = root_mean_squared_error(y_val, y_pred)
            else:
                score = np.nan # Metrica non riconosciuta

            score_results[metric].append(score)
            if verbose:
                print(f"    {metric.upper()}: {score:.4f}")

    # --- 5. RIEPILOGO FINALE ---
    if verbose:
        print("\n--- Nested Cross-Validation Final Report ---")

    final_summary = {}
    for metric, scores in score_results.items():
        mean_score = np.nanmean(scores)
        std_score = np.nanstd(scores)
        final_summary[metric] = {
            'mean': mean_score,
            'std': std_score,
            'all_scores': scores
        }
        if verbose:
            print(f"Final {metric.upper()} estimate: {mean_score:.4f} ± {std_score:.4f}")

    return {
        'performance_summary': final_summary,
        'all_fold_best_params': all_fold_best_params
    }

### Nested CV su Regressione logistica

In [47]:
from sklearn.linear_model import LogisticRegression

logisticRegression_params_grid = {
    'C': [0.5, 0.75, 1, 1.25, 1.5]
}

In [59]:
logisticRegression_result_nested_cv = nested_cv(
    model = LogisticRegression(),
    param_grid = logisticRegression_params_grid,
    X_train = X_train_embedded_lle,
    y_train = y_train,
    outer_splits = 5,
    inner_splits = 3,
    scoring = ['accuracy']
)


Performing Outer Fold 1/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'C': 1.5}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6302

Performing Outer Fold 2/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'C': 1.5}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6406

Performing Outer Fold 3/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'C': 1.5}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6146

Performing Outer Fold 4/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'C': 1.5}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6875

Performing Outer Fold 5/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'C': 1.5}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6458

--- Nested Cross-Validation Final Report ---
Fina

## Training sul test set

In [49]:
from collections import Counter
from sklearn.base import clone
import numpy as np

def train_final_model_from_nested_cv(model,
                                     all_fold_best_params,
                                     X, y,
                                     score_per_fold: list[float] = None,
                                     strategy: str = 'most_frequent',
                                     X_test=None, y_test=None,
                                     scoring: list[str] = None,
                                     verbose=True):
    """
    Allena un modello finale usando i migliori iperparametri ottenuti da una nested CV,
    e calcola le metriche su un test set opzionale se fornito.

    Parameters:
        model: modello sklearn
        all_fold_best_params: lista dei parametri ottimali per ogni fold
        X, y: dati completi per l'addestramento
        score_per_fold: punteggi dei fold esterni, richiesto per 'best_fold'
        strategy: 'most_frequent' o 'best_fold'
        X_test, y_test: test set opzionale per calcolare le metriche finali
        scoring: lista di metriche da calcolare (default auto)
        verbose: se True, stampa info

    Returns:
        final_model: modello allenato su tutto il dataset
        best_params: iperparametri usati
        test_metrics
    """
    if not all_fold_best_params:
        raise ValueError("La lista di best_params è vuota.")

    if strategy == 'most_frequent':
        # Conta la combinazione più ricorrente tra i dizionari
        param_counts = Counter([frozenset(p.items()) for p in all_fold_best_params])
        most_common_params = dict(param_counts.most_common(1)[0][0])
        if verbose:
            print(f"\n[STRATEGIA: most_frequent] Parametri più frequenti sui fold:")
            print(most_common_params)
        best_params = most_common_params

    elif strategy == 'best_fold':
        if score_per_fold is None:
            raise ValueError("score_per_fold è richiesto per la strategia 'best_fold'.")
        if len(score_per_fold) != len(all_fold_best_params):
            raise ValueError("score_per_fold e all_fold_best_params devono avere la stessa lunghezza.")

        best_index = int(np.nanargmax(score_per_fold))
        best_params = all_fold_best_params[best_index]

        if verbose:
            print(f"\n[STRATEGIA: best_fold] Selezionato il fold #{best_index + 1} con punteggio migliore: {score_per_fold[best_index]:.4f}")
            print(f"Parametri selezionati: {best_params}")

    else:
        raise ValueError("Strategia non supportata: usa 'most_frequent' o 'best_fold'.")

    # Clona il modello e imposta i parametri selezionati
    final_model = clone(model).set_params(**best_params)

    # Allena su tutto il dataset
    final_model.fit(X, y)

    test_metrics = {}
    # Calcolo delle metriche su test set, se fornito
    if X_test is not None and y_test is not None:
        if verbose:
            print("\nCalcolo delle metriche sul test set finale...")

        y_pred = final_model.predict(X_test)
        try:
            y_proba = final_model.predict_proba(X_test)[:, 1]
        except:
            y_proba = None

        # Determina task
        est_type = getattr(final_model, "_estimator_type", None)
        is_clf = (est_type == "classifier")

        if scoring is None:
            scoring = ['accuracy', 'roc_auc'] if is_clf else ['r2', 'mae', 'rmse']

        for metric in scoring:
            if metric == 'accuracy':
                test_metrics['accuracy'] = accuracy_score(y_test, y_pred)
            elif metric == 'roc_auc':
                if y_proba is not None:
                    test_metrics['roc_auc'] = roc_auc_score(y_test, y_proba)
                else:
                    test_metrics['roc_auc'] = np.nan
            elif metric == 'r2':
                test_metrics['r2'] = r2_score(y_test, y_pred)
            elif metric == 'mae':
                test_metrics['mae'] = mean_absolute_error(y_test, y_pred)
            elif metric == 'rmse':
                test_metrics['rmse'] = mean_squared_error(y_test, y_pred, squared=False)
            else:
                test_metrics[metric] = np.nan

        if verbose:
            for m, v in test_metrics.items():
                print(f"  {m.upper()}: {v:.4f}")

    return final_model, best_params, test_metrics


In [60]:
final_model_lr, best_params_lr, test_metrics_lr = train_final_model_from_nested_cv(
    model = LogisticRegression(),
    all_fold_best_params = logisticRegression_result_nested_cv['all_fold_best_params'],
    X = X_train_embedded_lle,
    y = y_train,
    X_test = X_test_embedded_lle,
    y_test = y_test,
    strategy = 'most_frequent',
    scoring = ['accuracy']
)


[STRATEGIA: most_frequent] Parametri più frequenti sui fold:
{'C': 1.5}

Calcolo delle metriche sul test set finale...
  ACCURACY: 0.6167


## Pipeline 2
   - Preprocessing: MinMaxScaler
   - Riduzione dimensionalità: Isomap (n_components=10)
   - Modello: RandomForestClassifier (ottimizzazione di n_estimators e max_depth)


### Scaling

In [54]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler(feature_range = (-1, 1)) # Scalo tra -1 e +1

X_train_min_max = min_max_scaler.fit_transform(X_train)

### Manifold Learning

In [55]:
from sklearn.manifold import Isomap

isomap_params_grid = {
    'n_neighbors': [5, 3, 7],
    'n_components': [2, 5, 10]
}

In [56]:
isomap_result_best_manifold = best_manifold(
    X = X_train_min_max,
    y = y_train,
    model = Isomap,
    param_grid = isomap_params_grid
)

Executing function--Params: {'n_neighbors': 3, 'n_components': 10} => AMI Score: 0.0216
Executing function--Params: {'n_neighbors': 3, 'n_components': 2} => AMI Score: 0.0271
Executing function--Params: {'n_neighbors': 7, 'n_components': 5} => AMI Score: 0.0282


In [63]:
best_score_isomap = isomap_result_best_manifold['best_score']
best_params_isomap = isomap_result_best_manifold['best_params']

print("Best score: ", best_score_isomap)
print("Best parameters: ", best_params_isomap)

Best score:  0.028198443657049595
Best parameters:  {'n_neighbors': 7, 'n_components': 5}


## Creazione dell'embedding

In [64]:
X_train_embedded_isomap = Isomap(
    n_neighbors =  best_params_isomap['n_neighbors'],
    n_components = best_params_isomap['n_components']
).fit_transform(X_train_min_max)

print(f"Shape of X_train after transformation: {X_train_embedded_isomap.shape}")

X_test_embedded_isomap = Isomap(
    n_neighbors =  best_params_isomap['n_neighbors'],
    n_components = best_params_isomap['n_components']
).fit_transform(X_test)

print(f"Shape of X_test after transformation: {X_test_embedded_isomap.shape}")

Shape of X_train after transformation: (960, 5)
Shape of X_test after transformation: (240, 5)


## RandomForestClassifier

In [65]:
from sklearn.ensemble import RandomForestClassifier

rf_params_grid = {
    'n_estimators': [100, 75, 150],
    'max_depth': [None, 10, 20, 30]
}
rf_result_nested_cv = nested_cv(
    model = RandomForestClassifier(),
    param_grid = rf_params_grid,
    X_train = X_train_embedded_isomap,
    y_train = y_train,
    outer_splits = 5,
    inner_splits = 3,
    scoring = ['accuracy']
)


Performing Outer Fold 1/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'max_depth': None, 'n_estimators': 75}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6823

Performing Outer Fold 2/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'max_depth': 20, 'n_estimators': 150}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6979

Performing Outer Fold 3/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'max_depth': 30, 'n_estimators': 100}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6510

Performing Outer Fold 4/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'max_depth': 20, 'n_estimators': 150}
  Calculating metrics on the outer test set...
    ACCURACY: 0.6875

Performing Outer Fold 5/5
Performing GridSearchCV (optimizing for 'accuracy')...
  Best Params for this fold: {'max_depth': 20

In [66]:
final_model_rf, best_params_rf, test_metrics_rf = train_final_model_from_nested_cv(
    model = RandomForestClassifier(),
    all_fold_best_params = rf_result_nested_cv['all_fold_best_params'],
    X = X_train_embedded_isomap,
    y = y_train,
    X_test = X_test_embedded_isomap,
    y_test = y_test,
    strategy = 'most_frequent',
    scoring = ['accuracy']
)


[STRATEGIA: most_frequent] Parametri più frequenti sui fold:
{'n_estimators': 150, 'max_depth': 20}

Calcolo delle metriche sul test set finale...
  ACCURACY: 0.5333
